In [1]:
from pyspark.sql import SparkSession


spark = SparkSession.builder.appName('churn').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/27 09:51:08 WARN Utils: Your hostname, aswin-Inspiron, resolves to a loopback address: 127.0.1.1; using 192.168.127.189 instead (on interface wlp1s0)
26/02/27 09:51:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 09:51:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/27 09:51:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
df = spark.read.csv('c1.csv', header=True, inferSchema=True)

In [3]:
df.show()

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|     OnlineSecurity|       OnlineBackup|   DeviceProtection|        TechSupport|        StreamingTV|    StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|  

In [4]:
df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



In [5]:
df.show()

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|     OnlineSecurity|       OnlineBackup|   DeviceProtection|        TechSupport|        StreamingTV|    StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|  

In [6]:
df = df.dropna()

In [7]:
from pyspark.sql.functions import col, trim, when

df = df.withColumn(
    "TotalCharges",
    when(trim(col("TotalCharges")) == "", None)
    .otherwise(trim(col("TotalCharges")))
)

df = df.withColumn("TotalCharges", col("TotalCharges").cast("double"))

df = df.withColumn("MonthlyCharges", col("MonthlyCharges").cast("double"))

In [8]:
df.printSchema()

root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: double (nullable = true)
 |-- Churn: string (nullable = true)



In [9]:
from pyspark.ml.feature import StringIndexer

label_indexer = StringIndexer(inputCol='Churn',outputCol='label')

In [10]:
categorical_cols = [
    "gender", "Partner", "Dependents", "PhoneService",
    "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport",
    "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod"
]

In [11]:
indexers = [ StringIndexer( inputCol= col, outputCol= col+'i') for col in categorical_cols]

In [12]:
df.show()

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|     OnlineSecurity|       OnlineBackup|   DeviceProtection|        TechSupport|        StreamingTV|    StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|  

In [13]:
numeric_cols = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

In [14]:
feature_cols = numeric_cols + [col + "i" for col in categorical_cols]

In [15]:
df = df.dropna(subset=[
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
])

In [16]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

In [17]:
from pyspark.ml.classification import LogisticRegression


lr = LogisticRegression(featuresCol='features', labelCol='label')

In [18]:
from pyspark.ml import Pipeline



pipeline = Pipeline(stages=indexers + [label_indexer, assembler, lr])

In [19]:
train, test = df.randomSplit([0.8,0.2], seed=42)

In [20]:
model = pipeline.fit(train)

26/02/27 09:51:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/02/27 09:51:37 WARN BlockManager: Asked to remove block broadcast_89, which does not exist


In [21]:
predictions = model.transform(test)

predictions.select("customerID", "label", "prediction", "probability").show(10)

+----------+-----+----------+--------------------+
|customerID|label|prediction|         probability|
+----------+-----+----------+--------------------+
|0004-TLHLJ|  1.0|       1.0|[0.34094183336184...|
|0013-SMEOE|  0.0|       0.0|[0.89394586197559...|
|0015-UOCOJ|  0.0|       0.0|[0.57524004711511...|
|0019-EFAEP|  0.0|       0.0|[0.90045221132836...|
|0023-HGHWL|  1.0|       1.0|[0.27141150021381...|
|0030-FNXPP|  0.0|       0.0|[0.80992156494376...|
|0042-RLHYP|  0.0|       0.0|[0.99732493180896...|
|0057-QBUQH|  0.0|       0.0|[0.96996038534215...|
|0078-XZMHT|  0.0|       0.0|[0.95794110205259...|
|0080-EMYVY|  0.0|       0.0|[0.88141566842581...|
+----------+-----+----------+--------------------+
only showing top 10 rows


In [22]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

eval = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')

In [23]:
auc = eval.evaluate(predictions)

In [24]:
print(auc)

0.8530501612571232
